# 11 — Negative Conditional Entropy

Meet the sharpest signature of entanglement: a conditional entropy that drops below zero — impossible for any classical pair.

**Prerequisites:** `02/10_quantum_mutual_information`.

**What you'll be able to do**
- Compute the quantum conditional entropy $S(A|B) = S(\rho_{AB}) - S(\rho_B)$.
- Show it is $-1$ bit for a Bell pair, where classically $H(X|Y) \ge 0$ always.
- Sweep a Werner state and watch both quantum effects switch off together as noise grows.

Classical conditional entropy is never negative — knowing $Y$ can only reduce your uncertainty about $X$. Quantum mechanics violates even this. For a Bell pair, $S(A|B) = -1$ bit. We compute it, then sweep a noisy Werner state to watch the negative region — the entangled zone — shrink and vanish.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.infotheory.quantum import quantum_mutual_information, quantum_conditional_entropy
from qot_course.quantum.composite import bell_state, tensor
from qot_course.quantum.density import density_matrix
from qot_course.quantum.states import KET_0, KET_PLUS

np.random.seed(42)
viz.use_course_style()

## 1. Negative conditional entropy

The quantum conditional entropy is

$$ S(A|B) = S(\rho_{AB}) - S(\rho_B). $$

Classically $H(X|Y) \ge 0$: knowing $Y$ can only reduce uncertainty about $X$, never make it negative. Quantum mechanics breaks this. For a Bell pair $S(\rho_{AB}) = 0$ (pure) but $S(\rho_B) = 1$ bit (maximally mixed reduced state), so $\mathbf{S(A|B) = -1}$ **bit** (Cerf & Adami, 1997). Operationally this negative number is the *coherent information* — the rate at which the shared entanglement lets you send quantum information back.

In [ ]:
rho_bell = density_matrix(bell_state())
rho_product = density_matrix(tensor(KET_0, KET_PLUS))
print(f"S(A|B)  Bell state    = {quantum_conditional_entropy(rho_bell, dims=[2, 2]):+.3f} bit  (impossible classically)")
print(f"S(A|B)  product state = {quantum_conditional_entropy(rho_product, dims=[2, 2]):+.3f} bit")
print()
print("Classical bound:  H(X|Y) >= 0  ALWAYS.")

## 2. The Werner-state sweep

Mix the Bell state with white noise:

$$ \rho(p) = (1 - p)\,|\mathrm{Bell}\rangle\langle\mathrm{Bell}| + p\,\frac{I_4}{4}, \qquad p \in [0, 1]. $$

At $p = 0$ we have the pure Bell pair; at $p = 1$ the maximally mixed two-qubit state. We plot both $I(A{:}B)$ and $S(A|B)$ across the sweep.

In [ ]:
def werner_state(p):
    return (1.0 - p) * density_matrix(bell_state()) + p * (np.eye(4, dtype=complex) / 4.0)

ps = np.linspace(0.0, 1.0, 60)
qmi = np.array([quantum_mutual_information(werner_state(p), dims=[2, 2]) for p in ps])
qce = np.array([quantum_conditional_entropy(werner_state(p), dims=[2, 2]) for p in ps])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharex=True)

axes[0].plot(ps, qmi, color=viz.SOURCE_COLOR, lw=2)
axes[0].axhline(1.0, color=viz.TARGET_COLOR, linestyle="--", label="classical max (1 bit)")
axes[0].axhline(0.0, color=COLORS["muted"], linewidth=0.8)
axes[0].set_xlabel("Werner noise parameter  p")
axes[0].set_ylabel("I(A:B)  (bits)")
axes[0].set_title("Quantum mutual information", pad=10)
axes[0].legend(loc="upper right")

axes[1].plot(ps, qce, color=viz.FLOW_COLOR, lw=2)
axes[1].axhline(0.0, color=viz.TARGET_COLOR, linestyle="--", label="classical floor (0)")
axes[1].fill_between(ps, qce, 0.0, where=qce < 0.0, color=viz.FLOW_COLOR, alpha=0.18,
                     label="negative region (entangled)")
axes[1].set_xlabel("Werner noise parameter  p")
axes[1].set_ylabel("S(A|B)  (bits)")
axes[1].set_title("Quantum conditional entropy", pad=10)
axes[1].legend(loc="lower right")

fig.suptitle("Werner-state sweep: two purely-quantum effects", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

**Read both panels.**

- **Left.** $I(A{:}B)$ starts at $2$ bits (pure Bell, double the classical maximum) and decays to $0$ as the state becomes the maximally mixed $I/4$. Everything above the dashed amber line ($1$ bit) is quantum.
- **Right.** $S(A|B)$ starts at $-1$ bit (negative — impossible classically) and rises through zero into the positive region. The shaded region is the **entangled zone**; once $S(A|B) \ge 0$, the state may be merely classically correlated.

Two purely-quantum effects, one sweep, shared origin — **entanglement**. As $p$ grows the resource is destroyed and both quantities relax to their classical regime.

*Caveat.* $S(A|B) \ge 0$ is *necessary* but not *sufficient* for separability — there exist "bound-entangled" states with non-negative conditional entropy. Entanglement is harder to detect than to break.

## Your turn

1. **The threshold.** Find numerically the smallest $p$ at which $S(A|B) \ge 0$. Compare to the Peres–Horodecki PPT bound $p \ge 2/3$ for separability — does negative conditional entropy track separability exactly, or is it a *strict* sub-criterion?
2. **MI crossing.** At what $p$ does $I(A{:}B)$ fall through the classical maximum of $1$ bit?
3. **Pure-state CMI.** For a pure entangled state $\cos\theta\,|00\rangle + \sin\theta\,|11\rangle$, show that $S(A|B) = -S(\rho_A) \le 0$ for all $\theta$.

## What you built

- You computed the quantum conditional entropy and found $-1$ bit for a Bell pair.
- You swept a Werner state and watched both quantum effects (excess MI, negative CMI) switch off as noise grew.
- You met the honest caveat: $S(A|B) \ge 0$ is necessary but not sufficient for separability (bound entanglement).

## What's next

The information arc has one piece left: a *distance* between density matrices. The synthesis `02/12_bures_distance` lifts the Fisher–Rao distance of `02/07` to the quantum setting — the Bures distance — and completes Movement 2, the spine of the course.

## References

- N. J. Cerf & C. Adami, "Negative entropy and information in quantum mechanics", *Physical Review Letters* 79, 5194 (1997). DOI:10.1103/PhysRevLett.79.5194
- M. M. Wilde, *Quantum Information Theory*, ch. 11, Cambridge University Press (2017).

**Previous:** `notebooks/02_InformationAndGeometry/10_quantum_mutual_information.ipynb` · **Next:** `notebooks/02_InformationAndGeometry/12_bures_distance.ipynb`